# Entrenamiento YOLOv8 — Detección cercana
**Clases:** carro · moto · persona · poste · senal_transito

Este notebook entrena YOLOv8 con tu dataset etiquetado desde Label Studio.

## 1. Instalación

## 2. Verificar GPU

In [6]:
import torch
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
device = '0' if torch.cuda.is_available() else 'cpu'
print('Usando:', device)

CUDA disponible: False
Usando: cpu


In [8]:
import torch
import sys
import os

# 1. Verificación básica
print(f"Versión de Python: {sys.version}")
print(f"Versión de PyTorch: {torch.__version__}")
print(f"¿CUDA detectado por PyTorch?: {torch.cuda.is_available()}")

# 2. Diagnóstico profundo si falla
if not torch.cuda.is_available():
    print("\n--- INICIANDO DIAGNÓSTICO DE HARDWARE ---")
    
    # Verificar si Windows ve la tarjeta NVIDIA
    if sys.platform == "win32":
        print("Ejecutando nvidia-smi para verificar drivers...")
        resultado = os.system("nvidia-smi")
        if resultado != 0:
            print("ERROR: 'nvidia-smi' no se reconoce. Los drivers de NVIDIA no están instalados o están mal configurados.")
    
    # Verificar si hay una versión de PyTorch incorrecta
    if "cpu" in torch.__version__:
        print("CAUSA DETECTADA: Tienes instalada la versión de PyTorch 'CPU-only'.")
        print("SOLUCIÓN: Debes reinstalar con el index-url de CUDA 12.1.")
else:
    print(f"\n¡ÉXITO! GPU detectada: {torch.cuda.get_device_name(0)}")

Versión de Python: 3.11.2 (tags/v3.11.2:878ead1, Feb  7 2023, 16:38:35) [MSC v.1934 64 bit (AMD64)]
Versión de PyTorch: 2.11.0+cpu
¿CUDA detectado por PyTorch?: False

--- INICIANDO DIAGNÓSTICO DE HARDWARE ---
Ejecutando nvidia-smi para verificar drivers...
CAUSA DETECTADA: Tienes instalada la versión de PyTorch 'CPU-only'.
SOLUCIÓN: Debes reinstalar con el index-url de CUDA 12.1.


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt")

cap = cv2.VideoCapture(0)
print("Cámara iniciada — presiona Q para salir")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.15, verbose=False)[0]
    annotated = results.plot()

    cv2.imshow("Deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Cámara iniciada — presiona Q para salir


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt")

cap = cv2.VideoCapture(0)
print("Cámara iniciada — presiona Q para salir")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.15, verbose=False)[0]
    annotated = results.plot()

    cv2.imshow("Deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Cámara iniciada — presiona Q para salir


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt")

cap = cv2.VideoCapture(0)
print("Cámara iniciada — presiona Q para salir")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.15, verbose=False)[0]
    annotated = results.plot()

    cv2.imshow("Deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Cámara iniciada — presiona Q para salir


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt")

cap = cv2.VideoCapture(0)
print("Cámara iniciada — presiona Q para salir")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.15, verbose=False)[0]
    annotated = results.plot()

    cv2.imshow("Deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Cámara iniciada — presiona Q para salir


## 3. Verificar dataset

In [22]:
from pathlib import Path

dataset_path = Path('./dataset_yolov8_listo/yolo_dataset')  # ruta correcta donde Label Studio exportó

train_imgs = list((dataset_path / 'images' / 'train').glob('*.jpg'))
val_imgs   = list((dataset_path / 'images' / 'val').glob('*.jpg'))
train_lbl  = list((dataset_path / 'labels' / 'train').glob('*.txt'))
val_lbl    = list((dataset_path / 'labels' / 'val').glob('*.txt'))

print(f'Train: {len(train_imgs)} imágenes, {len(train_lbl)} etiquetas')
print(f'Val  : {len(val_imgs)} imágenes, {len(val_lbl)} etiquetas')
assert len(train_imgs) == len(train_lbl), '⚠️ Desbalance en train'
assert len(val_imgs)   == len(val_lbl),   '⚠️ Desbalance en val'
print('✅ Dataset OK')

Train: 96 imágenes, 96 etiquetas
Val  : 24 imágenes, 24 etiquetas
✅ Dataset OK


## 4. Entrenamiento

In [ ]:
from ultralytics import YOLO

# Modelo base — yolov8n (nano) es el más rápido y ligero
# Opciones: yolov8n, yolov8s, yolov8m, yolov8l, yolov8x
model = YOLO('yolov8n.pt')

results = model.train(
    data      = str(dataset_path / 'data.yaml'),
    epochs    = 100,          # empieza con 100, sube si el modelo no converge
    imgsz     = 640,
    batch     = 16,           # baja a 8 si hay error de memoria
    device    = device,
    patience  = 20,           # early stopping si no mejora en 20 épocas
    save      = True,
    project   = 'runs/detect',
    name      = 'deteccion_cercana',
    exist_ok  = True,
    pretrained= True,         # transfer learning desde COCO
    optimizer = 'AdamW',
    lr0       = 0.001,
    augment   = True,         # data augmentation automático
    verbose   = True
)

print('✅ Entrenamiento finalizado')
print(f'Mejor modelo: {results.save_dir}/weights/best.pt')

Ultralytics 8.4.46  Python-3.11.2 torch-2.11.0+cpu CPU (AMD Ryzen 7 7735HS with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_yolov8_listo\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=deteccion_cercana, nbs=64, nms=False, opset=None, optimize=False, optimizer

## 5. Evaluación en validación

In [ ]:
metrics = model.val()
print(f'mAP50     : {metrics.box.map50:.3f}')
print(f'mAP50-95  : {metrics.box.map:.3f}')
print(f'Precisión : {metrics.box.mp:.3f}')
print(f'Recall    : {metrics.box.mr:.3f}')

Ultralytics 8.4.46  Python-3.11.2 torch-2.11.0+cpu CPU (AMD Ryzen 7 7735HS with Radeon Graphics)
Model summary (fused): 73 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 550.4103.6 MB/s, size: 126.0 KB)
val: Scanning C:\Users\coral\OneDrive\Documentos\pythonproyect\PROYECTO_DETECION_CERCANA\labels\val.cache... 24 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 24/24  0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 2.2s/it 4.4s<11.5s
                   all         24         73      0.797      0.712      0.809      0.415
                 carro         11         16      0.756      0.812      0.875      0.546
                  moto          6          6      0.748      0.667      0.776      0.337
               persona         12         21      0.949      0.885      0.964      0.541
                 poste         11         12      0.683      0.417      0.566  

## 6. Prueba con una imagen

In [ ]:
import random
from PIL import Image
import matplotlib.pyplot as plt

# Tomar imagen aleatoria de validación
test_img = random.choice(list((dataset_path / 'images' / 'val').glob('*.jpg')))

# Usar el path correcto desde results.save_dir
best_model_path = results.save_dir / 'weights' / 'best.pt'
best_model = YOLO(str(best_model_path))
pred = best_model.predict(str(test_img), conf=0.25, save=False)[0]

img_plot = pred.plot()  # imagen con bboxes dibujados
plt.figure(figsize=(10, 8))
plt.imshow(img_plot[..., ::-1])
plt.axis('off')
plt.title(f'Predicción: {test_img.name}')
plt.show()

for box in pred.boxes:
    cls_id = int(box.cls)
    conf   = float(box.conf)
    name   = best_model.names[cls_id]
    print(f'  {name}: {conf:.2f}')


image 1/1 c:\Users\coral\OneDrive\Documentos\pythonproyect\PROYECTO_DETECION_CERCANA\dataset_yolov8_listo\yolo_dataset\images\val\a3018c62-img108.jpg: 640x640 1 carro, 138.1ms
Speed: 4.8ms preprocess, 138.1ms inference, 6.2ms postprocess per image at shape (1, 3, 640, 640)


<Figure size 1000x800 with 1 Axes>

  carro: 0.53


## 7. Exportar modelo
Exporta a ONNX para despliegue en producción o a TFLite para móviles.

In [ ]:
# Exportar a ONNX (compatible con casi cualquier plataforma)
best_model.export(format='onnx', imgsz=640)
print('✅ Modelo exportado a ONNX')
print(f'   Ubicación: {results.save_dir}')

# Descomenta para exportar a TFLite (móvil/edge)
# best_model.export(format='tflite', imgsz=640)

Ultralytics 8.4.46  Python-3.11.2 torch-2.11.0+cpu CPU (AMD Ryzen 7 7735HS with Radeon Graphics)

PyTorch: starting from 'C:\Users\coral\OneDrive\Documentos\pythonproyect\PROYECTO_DETECION_CERCANA\runs\detect\runs\detect\deteccion_cercana\weights\best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
   ---------------------------------------- 0.0/16.4 MB ? eta -:--:--
   --------------- ------------------------ 6.6/16.4 MB 40.1 MB/s eta 0:00:01
   -------------------------------------- - 15.7/16.4 MB 41.2 MB/s eta 0:00:01
   ---------------------------------------- 16.4/16.4 MB 41.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.9 MB ? eta -:--:--
   ---------------------------------------- 12.9/12.9 MB 67.2 MB/s  0:00:00

   ---------------------------------------- 0/3 [onnxruntime]
   --------------

In [3]:
import cv2
from ultralytics import YOLO

model = YOLO(r"runs\detect\runs\detect\deteccion_cercana\weights\best.pt")

cap = cv2.VideoCapture(0)
print("Cámara iniciada — presiona Q para salir")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.predict(frame, conf=0.15, verbose=False)[0]
    annotated = results.plot()

    cv2.imshow("Deteccion", annotated)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Cámara iniciada — presiona Q para salir


In [ ]:
cv2.destroyAllWindows()